# Clone đúng bản DyGEnc vào Kaggle

Bật **Internet** và thêm dataset **agqa-balanced** bằng Add Input. Cell này chỉ tải source code, không copy dataset/model và không chạy Slurm. Tag `kaggle-ram-v1` được dùng để cố định phiên bản.

Nếu repo riêng tư, đặt `PRIVATE_REPO = True`, tạo Kaggle Secret tên `GITHUB_TOKEN` có quyền đọc Contents của repo và bật quyền truy cập secret cho notebook. Không dán token vào code/URL.

Clone xong chưa có nghĩa là train được: runner hiện tại cần một GPU native BF16; T4/P100 cần được điều chỉnh trước khi huấn luyện.

In [ ]:
from pathlib import Path
import base64
import os
import subprocess

REPO_URL = 'https://github.com/tdat1465/DyGEnc.git'
REF = 'kaggle-ram-v1'
REPO_DIR = Path('/kaggle/working/DyGEnc')
PRIVATE_REPO = False

git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
# Keep optional authentication only in child-process environment.
for key in list(git_env):
    if key.startswith('GIT_TRACE') or key.startswith('GIT_CONFIG_') or key == 'GIT_CURL_VERBOSE':
        git_env.pop(key, None)
if PRIVATE_REPO:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    auth = base64.b64encode(('x-access-token:' + token).encode()).decode()
    git_env.update({
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + auth,
    })
    del token, auth

def git(*args):
    return subprocess.run(['git', *map(str, args)], env=git_env,
                          check=True, text=True, stdout=subprocess.PIPE).stdout.strip()

try:
    if not REPO_DIR.exists():
        git('clone', '--depth', '1', '--branch', REF, REPO_URL, REPO_DIR)
    else:
        if not (REPO_DIR / '.git').is_dir():
            raise RuntimeError(f'{REPO_DIR} đã tồn tại nhưng không phải Git checkout; chọn thư mục khác.')
        if git('-C', REPO_DIR, 'remote', 'get-url', 'origin') != REPO_URL:
            raise RuntimeError('Thư mục hiện tại thuộc repo khác; không ghi đè.')
        if git('-C', REPO_DIR, 'status', '--porcelain'):
            raise RuntimeError('Checkout có thay đổi chưa lưu; giữ lại hoặc chọn REPO_DIR khác.')
        git('-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', 'tag', REF)
    git('-C', REPO_DIR, 'checkout', '--detach', f'refs/tags/{REF}')
    commit = git('-C', REPO_DIR, 'rev-parse', 'HEAD')
    assert (REPO_DIR / 'src/server_train.py').is_file(), 'Thiếu runner mới.'
    print('Code:', REPO_DIR)
    print('Commit:', commit)
finally:
    # Do not retain the authorization header in the notebook namespace.
    git_env.clear()

os.chdir(REPO_DIR)
